# SWOT L2 virtualizarr example - Icechunk store
### (NASA CMR Shortname: SWOT_L2_LR_SSH_Basic_D)

#### *Author: Ed Armstrong, PO.DAAC*

*Reference herein to any specific commercial product, process, or service by trade name, trademark, manufacturer, or otherwise, does not constitute or imply its endorsement by the United States Government or the Jet Propulsion Laboratory, California Institute of Technology.*

This notebook is the Icechunk counterpart of the kerchunk notebook (`virtualizarr_recipes-SWOT_L2_LR_SSH_Basic_D`). It follows the same recipe so the two stores are equivalent:

1. One reference per granule (only `num_lines` and `num_pixels` are loaded into memory, everything else stays virtual).
2. The granules are stacked along a **new `granule` dimension**.  This is a monotonic index. SWOT L2 `time` is a 2-D variable on `num_lines`, so it can **not** be used as the concat dimension.
3. `latitude` / `longitude` are concatenated along `granule`; all other coordinates are taken from the first granule (`compat="override"`).
4. `orbit`, `cycle`, `ppass` and `filename` are added as coordinates on `granule`.

Differences from the kerchunk notebook are only those required by VirtualiZarr 2.x / Icechunk: parser + object-store registry instead of `reader_options`, `open_virtual_mfdataset` instead of `open_virtual_dataset` + `xr.concat`, and `.vz.to_icechunk()` instead of `.virtualize.to_kerchunk()`.

Currently drops the `polarization_karin` variable because it composed of characters.

Some earlier SWOT granules have inconsistent `num_lines` dimensions.  These are ignored for now.

Reference notebook for HTTPS access with VirtualiZarr: https://notebooksharing.space/view/a0214376374af43776a54bf69b0b0308bbc0499fad345fcb5eb70e9ad9cf46c6

In [ ]:
# Built-in packages
import os
import sys
import warnings
import shutil
import multiprocessing
from pathlib import Path
from urllib.parse import urlparse

# Filesystem management
import fsspec
import earthaccess

# Data handling
import numpy as np
import pandas as pd
import xarray as xr
import zarr

# Virtualizarr / Icechunk / obstore
import virtualizarr as vz
import icechunk
import obstore
from obstore.auth.earthdata import NasaEarthdataCredentialProvider  # S3 access (in-region only)
from obstore.store import S3Store
from virtualizarr.parsers import HDFParser
try:
    from obspec_utils.registry import ObjectStoreRegistry
except ImportError:  # older virtualizarr
    from virtualizarr.registry import ObjectStoreRegistry

# Parallel computing
import dask
from dask.distributed import Client, LocalCluster

# Coiled for Dask management (only needed for the Coiled option in section 2.1)
from coiled import Cluster as CoiledCluster

for _ in (earthaccess, vz, zarr, xr, obstore, dask, fsspec, icechunk):
    print(f"{_.__name__}: {_.__version__}")

## 1. Get Data File S3 endpoints in Earthdata Cloud
The first step is to find the endpoints to the files. Handling access credentials to Earthdata and then finding the endpoints is done with the `earthaccess` package.

In [2]:
# Get Earthdata creds
auth = earthaccess.login()

In [3]:
shortname = "SWOT_L2_LR_SSH_EXPERT_D"

# Set to False to process the full record (as in the kerchunk notebook). True = small test subset.
SUBSET = False

if SUBSET:
    pgd_range = ("2024-07-26", "2024-08-08")
    pid_range = ("2025-05-06", "2025-05-22")
else:
    pgd_range = ("2023-07-26", "2025-04-08")
    pid_range = ("2025-05-06", "2026-11-01")


# pgd_range = ("2023-03-27", "2025-04-08") problem with earlier granules
# date earlier than 20230527 are formatted as  {'num_lines': 9865, 'num_pixels': 69}  
# (num_lines one one pixel smaller) and granules between 20230527 and 20230726 are a mix of these dimensions.


In [4]:
# Locate SWOT L2 Version D file information / metadata for the PGD files.
# The start of the science orbits files is 2023-07-26 (cycle 1)
granule_info_1 = earthaccess.search_data(
    short_name=shortname,
    granule_name="SWOT_L2_LR_SSH_EXPERT*PGD*.nc",
    #granule_name="SWOT_L2_LR_SSH_*PGD*.nc",
    temporal=pgd_range,
    )

In [5]:
# Locate SWOT L2 Version D file information / metadata for the PID files:
granule_info_2 = earthaccess.search_data(
    short_name=shortname,
    granule_name="SWOT_L2_LR_SSH_EXPERT*PID*.nc",
    #granule_name="SWOT_L2_LR_SSH_*PID*.nc",
    temporal=pid_range,
    )

In [6]:
# append the two granule lists. They are sorted in time.
granule_info = granule_info_1 + granule_info_2
print("Number of files found: ", len(granule_info))

Number of files found:  27010


In [7]:
# Get S3 (direct) and HTTPS endpoints for all files. Both lists are in the same order as granule_info.
# NOTE: these names are never re-assigned below (the old notebook overwrote `data_links` inside a loop).
s3_links    = [g.data_links(access="direct")[0] for g in granule_info]
https_links = [g.data_links(access="https")[0]  for g in granule_info]
print(len(s3_links))
print(s3_links[0:2])
print(https_links[0:2])

bucket = urlparse(s3_links[0]).netloc
print("bucket:", bucket)

27010
['s3://podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Expert_001_134_20230726T000155_20230726T000155_PGD0_02.nc', 's3://podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Expert_001_135_20230726T005326_20230726T005326_PGD0_02.nc']
['https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Expert_001_134_20230726T000155_20230726T000155_PGD0_02.nc', 'https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Expert_001_135_20230726T005326_20230726T005326_PGD0_02.nc']
bucket: podaac-swot-ops-cumulus-protected


## 2. Generate the combined virtual dataset

### 2.1 Start a Dask cluster
Use a local cluster (as in the kerchunk notebook), or Coiled for the whole record. Create **one** cluster only.

If using an `m6i.4xlarge` AWS EC2 instance there are 16 CPUs available; change `n_workers` as needed for a different VM-type.

In [8]:
# Check how many cpu's are on this VM:
print("CPU count =", multiprocessing.cpu_count())

CPU count = 18


In [12]:
cloud_opts = {
    "region": "us-west-2",
    "worker_vm_types": ["m6i.xlarge"],
    "scheduler_vm_types": ["m6i.xlarge"],
    "spot_policy": "spot_with_fallback",
    "name": "swot-virtualization",
    "environ": {"EARTHDATA_TOKEN": auth.token["access_token"]},
}

def create_dask_cluster(environment="local", n_workers=4, cloud_opts={}):
    if environment == "local":
        print("Creating new local Dask client")
        cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1, silence_logs=40)
    else:
        print("Creating new Coiled Dask client")
        cluster = CoiledCluster(n_workers=n_workers, **cloud_opts)
    client = Client(cluster)
    return client, cluster

if "client" not in locals() or "cluster" not in locals():
    client, cluster = create_dask_cluster(environment="coiled", n_workers=32, cloud_opts=cloud_opts)

dask.config.set({"distributed.worker.connections.outgoing": 5})
print(client.cluster)
print("View any work being done on the cluster here", client.dashboard_link)

Creating new Coiled Dask client


Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                               ╷                                              │
│   Package                     │ Note                                         │
│ ╶─────────────────────────────┼────────────────────────────────────────────╴ │
│   coiled_local_Downloads      │ Source wheel built from ~/Downloads          │
│                               ╵                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                          ╷                                       ╷           │
│   Package                │ Error                                 │ Risk      │
│ ╶────────────────────────┼───────────────────────────────────────┼─────────╴ │
│   podaac-data-subscriber │ Pip check had the following issues    │ Warning   │
│                          │ that need resolving:                  │           │
│                          │ podaac-data-subscriber 1.15.2 has     │           │
│                          │ requirement packaging<24.0,>=23.0,    │           │
│                          │ but you have packaging 25.0.          │           │
│                          ╵                                       ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

Cluster(swot-virtualization, 'tls://44.249.235.234:443', workers=2, threads=8, memory=28.69 GiB)
View any work being done on the cluster here https://cluster-pytzn.dask.host/woRxNb1vxb9jWLz6/status


/Users/earmstro/.local/lib/python3.12/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.4  | 4.4.5     | 4.4.5   |
| numpy   | 2.2.4  | 2.3.1     | 2.3.1   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


### 2.2 Create the combined VDS (S3 endpoints)
Equivalent of the kerchunk notebook's `open_virtual_dataset(..., loadable_variables=coord_vars)` per file followed by `xr.concat(dim="granule", ...)`.

In [13]:
# Same settings as the kerchunk notebook:
coord_vars  = ["num_lines", "num_pixels"]   # loaded into memory ('loadable_variables')
coords_list = ["latitude", "longitude"]     # coordinates that are concatenated along 'granule'

# Object store registry with Earthdata credentials (S3 direct access, in-region only)
credentials_endpoint = "https://archive.swot.podaac.earthdata.nasa.gov/s3credentials"
s3_store = S3Store(
    bucket=bucket,
    region="us-west-2",
    credential_provider=NasaEarthdataCredentialProvider(credentials_endpoint, auth=auth.token["access_token"]),
    virtual_hosted_style_request=False,
    client_options={"allow_http": True},
)
registry = ObjectStoreRegistry({f"s3://{bucket}": s3_store})

In [15]:
%%time
# Create the individual references (in parallel on the Dask cluster) and combine them along a NEW 'granule' dimension.
# - concat_dim="granule": SWOT L2 'time' is a 2-D variable, so concatenating on "time" fails
#   with "time already exists as coordinate or variable name".
# - data_vars="all": every data variable gets the 'granule' dimension (same as the xr.concat default in the kerchunk notebook).
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Numcodecs codecs*", category=UserWarning)
    vds_s3 = vz.open_virtual_mfdataset(
        urls=s3_links,
        registry=registry,
        #parser=HDFParser(),
        parser=HDFParser(drop_variables=["polarization_karin"]),
        combine="nested",
        concat_dim="granule",
        parallel="dask",
        data_vars="all",
        coords=coords_list,
        compat="override",
        combine_attrs="drop_conflicts",
        loadable_variables=coord_vars,
        decode_times=False
    )
print(vds_s3)

<xarray.Dataset> Size: 4TB
Dimensions:                                (granule: 27010, num_lines: 9866,
                                            num_pixels: 69, num_sides: 2)
Coordinates:
    latitude_nadir                         (num_lines) int32 39kB ManifestArr...
    longitude_nadir                        (num_lines) int32 39kB ManifestArr...
    latitude                               (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
    longitude                              (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
Dimensions without coordinates: granule, num_lines, num_pixels, num_sides
Data variables: (12/98)
    time                                   (granule, num_lines) float64 2GB M...
    time_tai                               (granule, num_lines) float64 2GB M...
    ssh_karin                              (granule, num_lines, num_pixels) int

### 2.3 Add granule-level coordinates
Extract a start time for each granule, plus the cycle and pass number, and the netCDF filename (identical to the kerchunk notebook).

In [16]:
orbit_start = []
cycle_num = []
pass_num = []
file_list = []

for g, link in zip(granule_info, https_links):
    datetime_str = g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']
    orbit_start.append(np.datetime64(datetime_str.rstrip("Z")))   # strip 'Z' to avoid the timezone warning
    track = g['umm']['SpatialExtent']['HorizontalSpatialDomain']['Track']
    cycle_num.append(track['Cycle'])
    pass_num.append(track['Passes'][0]['Pass'])
    file_list.append(Path(link).name)

In [17]:
# Add the new concat dimension of "granule" that are sequential numbers
granule_index = np.arange(len(s3_links))

assert vds_s3.sizes["granule"] == len(granule_index), "granule count does not match number of files"

vds_s3 = vds_s3.assign_coords(
    granule=("granule", granule_index),
    orbit=("granule", orbit_start),
    cycle=("granule", cycle_num),
    ppass=("granule", pass_num),
    filename=("granule", file_list),
)

In [18]:
vds_s3

<xarray.Dataset> Size: 4TB
Dimensions:                                (granule: 27010, num_lines: 9866,
                                            num_pixels: 69, num_sides: 2)
Coordinates:
  * granule                                (granule) int64 216kB 0 1 ... 27009
    orbit                                  (granule) datetime64[ms] 216kB 202...
    cycle                                  (granule) int64 216kB 1 1 1 ... 49 49
    ppass                                  (granule) int64 216kB 134 135 ... 427
    filename                               (granule) <U72 8MB 'SWOT_L2_LR_SSH...
    latitude_nadir                         (num_lines) int32 39kB ManifestArr...
    longitude_nadir                        (num_lines) int32 39kB ManifestArr...
    latitude                               (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
    longitude                              (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
Dimensions without coordinates: num_lines, num_pixels, num_sides
Data variables: (12/98)
    time                                   (granule, num_lines) float64 2GB M...
    time_tai                               (granule, num_lines) float64 2GB M...
    ssh_karin                              (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
    ssh_karin_qual                         (granule, num_lines, num_pixels) uint32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=uint32, chunks=(1,...
    ssh_karin_uncert                       (granule, num_lines, num_pixels) uint16 37GB ManifestArray<shape=(27010, 9866, 69), dtype=uint16, chunks=(1,...
    ssha_karin                             (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
    ...                                     ...
    swh_ssb_cor_source                     (granule, num_lines, num_pixels) uint8 18GB ManifestArray<shape=(27010, 9866, 69), dtype=uint8, chunks=(1, ...
    swh_ssb_cor_source_2                   (granule, num_lines, num_pixels) uint8 18GB ManifestArray<shape=(27010, 9866, 69), dtype=uint8, chunks=(1, ...
    wind_speed_ssb_cor_source              (granule, num_lines, num_pixels) uint8 18GB ManifestArray<shape=(27010, 9866, 69), dtype=uint8, chunks=(1, ...
    wind_speed_ssb_cor_source_2            (granule, num_lines, num_pixels) uint8 18GB ManifestArray<shape=(27010, 9866, 69), dtype=uint8, chunks=(1, ...
    volumetric_correlation                 (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
    volumetric_correlation_uncert          (granule, num_lines, num_pixels) uint32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=uint32, chunks=(1,...
Attributes: (12/22)
    Conventions:                      CF-1.7
    title:                            Level 2 Low Rate Sea Surface Height Dat...
    institution:                      CNES
    source:                           Ka-band radar interferometer
    platform:                         SWOT
    reference_document:               D-56407_SWOT_Product_Description_L2_LR_SSH
    ...                               ...
    xref_param_l2_lr_precalssh_file:  SWOT_Param_L2_LR_PreCalSSH_20000101T000...
    xref_reforbittrack_files:         SWOT_RefOrbitTrack125mPass1_Nom_2000010...
    xref_geco_database_version:       v107
    ellipsoid_semi_major_axis:        6378137.0
    ellipsoid_flattening:             0.0033528106647474805
    references:                       V1.4.1

In [19]:
pd.Index(orbit_start).is_monotonic_increasing

True

### 2.4 Duplicate to create a VDS with HTTPS endpoints
Instead of rewriting the S3 URLs by string manipulation, derive the HTTPS base from the endpoints `earthaccess` returned (the SWOT HTTPS archive host differs from the generic PO.DAAC one), and verify the mapping for every file.

In [20]:
# key = path inside the bucket, e.g. 'SWOT_L2_LR_SSH_D/SWOT_L2_LR_SSH_Basic_..._01.nc'
s3_prefix = f"s3://{bucket}/"
key0 = s3_links[0][len(s3_prefix):]
assert https_links[0].endswith(key0)
https_base = https_links[0][: -len(key0)]       # e.g. https://archive.swot.podaac.earthdata.nasa.gov/<bucket>/
print("HTTPS base:", https_base)

def s3_to_http_url(old_s3_path: str) -> str:
    """Change S3 endpoint to HTTPS."""
    return https_base + old_s3_path[len(s3_prefix):]

# Every mapped URL must equal the HTTPS link that earthaccess reports
assert all(s3_to_http_url(s) == h for s, h in zip(s3_links, https_links))

HTTPS base: https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/


In [21]:
vds_http = vds_s3.vz.rename_paths(s3_to_http_url)
print(vds_http)

<xarray.Dataset> Size: 4TB
Dimensions:                                (granule: 27010, num_lines: 9866,
                                            num_pixels: 69, num_sides: 2)
Coordinates:
  * granule                                (granule) int64 216kB 0 1 ... 27009
    orbit                                  (granule) datetime64[ms] 216kB 202...
    cycle                                  (granule) int64 216kB 1 1 1 ... 49 49
    ppass                                  (granule) int64 216kB 134 135 ... 427
    filename                               (granule) <U72 8MB 'SWOT_L2_LR_SSH...
    latitude_nadir                         (num_lines) int32 39kB ManifestArr...
    longitude_nadir                        (num_lines) int32 39kB ManifestArr...
    latitude                               (granule, num_lines, num_pixels) int32 74GB ManifestArray<shape=(27010, 9866, 69), dtype=int32, chunks=(1, ...
    longitude                              (granule, num_lines, num_pixels) int32 74GB Man

## 3. Save to Icechunk stores
One store each for the S3 and HTTPS endpoints. The virtual chunk container prefix must cover the URLs stored in the references (Icechunk validates this when writing).

In [22]:
def create_local_icechunk_repo_httpaccess(repo_name: str, vcc_http_base: str, overwrite: bool = False):
    """
    Creates an icechunk repository locally, with pointers to data chunks in HTTPS locations.

    Inputs
    ------
    * repo_name (str): path to icechunk repo.
    * vcc_http_base (str): URL prefix of all HTTPS chunk locations (must end with '/').
    * overwrite (bool): delete an existing repo at repo_name first (needed when re-running a cell).
    """
    if overwrite and os.path.exists(repo_name):
        shutil.rmtree(repo_name)
    storage = icechunk.local_filesystem_storage(path=repo_name)
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(
            url_prefix=vcc_http_base,
            store=icechunk.http_store(),
        )
    )
    return icechunk.Repository.create(storage, config)

In [23]:
def create_local_icechunk_repo_s3access(repo_name: str, vcc_bucket: str, overwrite: bool = False):
    """
    Creates an icechunk repository locally, with pointers to data chunks in S3 locations.

    Inputs
    ------
    * repo_name (str): path to icechunk repo.
    * vcc_bucket (str): 's3://<bucket>' that holds the data files.
    * overwrite (bool): delete an existing repo at repo_name first (needed when re-running a cell).
    """
    if overwrite and os.path.exists(repo_name):
        shutil.rmtree(repo_name)
    storage = icechunk.local_filesystem_storage(path=repo_name)
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(
            url_prefix=vcc_bucket + "/",
            store=icechunk.s3_store(region="us-west-2", anonymous=True),
        )
    )
    return icechunk.Repository.create(storage, config)

#### S3 endpoints

In [24]:
# Create icechunk repo and session
fname_store_s3access = "SWOT_L2_LR_SSH_EXPERT_D.icechunk_v2.s3"
repo = create_local_icechunk_repo_s3access(fname_store_s3access, "s3://" + bucket, overwrite=True)
session = repo.writable_session("main")

  2026-11-12T19:29:29.415940Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329

  2026-11-12T19:29:29.466788Z  WARN icechunk_storage::readback: conditional PUT is enabled but `unsafe_use_metadata` is disabled — lost-response recovery for conditional writes requires user metadata to stamp write-ids; without it, transient PUT failures may surface as spurious conflicts even when the write actually landed. See icechunk_storage::Settings::unsafe_use_metadata.
    at icechunk-storage/src/readback.rs:28



In [25]:
# Then write to the store (.vz is the VirtualiZarr 2.x accessor; .virtualize is the old v1 name):
vds_s3.vz.to_icechunk(session.store)
session.commit("Initial commit.")

/Users/earmstro/miniforge3/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=72, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


'ZSNVP3HNGXZK6WQVF8Y0'

#### HTTPS endpoints

In [26]:
# Create icechunk repo and session
fname_store_httpaccess = "SWOT_L2_LR_SSH_EXPERT_D.icechunk_v2.https"
repo = create_local_icechunk_repo_httpaccess(fname_store_httpaccess, https_base, overwrite=True)
session = repo.writable_session("main")

  2026-11-12T19:29:44.872604Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329



In [27]:
# Then write to the store:
vds_http.vz.to_icechunk(session.store)
session.commit("Initial commit.")

/Users/earmstro/miniforge3/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=72, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


'ASG5WPFEGAS73Z2NEDJG'

In [28]:
!tar -cvf $fname_store_s3access".tar" $fname_store_s3access

a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/snapshots
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/chunks
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/transactions
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/repo
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/overwritten
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/overwritten/repo.30709168220769.16KJDB3B4DCD384RBEW0
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/ZAAGV2GFFWMKJPD0DT80
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/H53G80GVS86MDEEY3X80
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/EYM7R267TJZ6B0MD3630
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/ZC34MZ88A976KZAA9PCG
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/T3807WMR55DAPD6ZTHAG
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/CMRVGNQGJ056VE80EVFG
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/8BCCS3QXGADTZPAM76FG
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.s3/manifests/V9A7CNBXCBH33M33NH90
a 

In [29]:
!tar -cvf $fname_store_httpaccess".tar" $fname_store_httpaccess

a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/snapshots
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/chunks
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/transactions
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/repo
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/overwritten
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/overwritten/repo.30709168208893.4SSREJHC9R2HCBB6PV9G
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests/JQQVXXPQ243840WVT2JG
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests/YJSA8X84Z6X3TXDQXWTG
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests/BARBKXFE7FN5KNWB523G
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests/78JMFDH094QQ0Z55QE7G
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests/RBFAXEKX914XWW9F2H00
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests/A57KRW4G4WE45GB0HW2G
a SWOT_L2_LR_SSH_Basic_D.icechunk_v2.https/manifests/QBSYJGSZFGG6XHHR8DM0
a SWOT_L2_LR_SSH_Basic_D.ice

## 4. Quick check of the S3 store (optional, in-region)
Open the store read-only and confirm it matches the kerchunk reference (dimensions `granule`, `num_lines`, `num_pixels`).

In [30]:
# Chunks are in a protected bucket, so authorize the virtual chunk container with Earthdata S3 credentials.
#creds = earthaccess.get_s3_credentials(endpoint=credentials_endpoint)

# 2. Call get_s3_credentials on the auth object directly
auth = earthaccess.login(strategy="interactive", persist=True)
creds = auth.get_s3_credentials(
    endpoint=credentials_endpoint
)

cred_map = {s3_prefix: icechunk.s3_credentials(
    access_key_id=creds["accessKeyId"], secret_access_key=creds["secretAccessKey"],
    session_token=creds["sessionToken"])}

repo_r = icechunk.Repository.open(
    icechunk.local_filesystem_storage(path=fname_store_s3access),
    authorize_virtual_chunk_access=cred_map,
)
ds_check = xr.open_zarr(repo_r.readonly_session("main").store, consolidated=False, zarr_format=3)
ds_check

  2026-11-12T19:31:22.109954Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329



TypeError: 'S3Credentials.Static' object is not an instance of 'Credentials'

In [31]:
# shutdown cluster
client.shutdown()